In [ ]:


import os
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
from google.colab import drive

print("Mounting Drive and Loading Data...")
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Datastorm'
silver_path = os.path.join(base_path, 'Silver')
bronze_path = os.path.join(base_path, 'Bronze')
gold_path = os.path.join(base_path, 'Gold')

# Load the clean shop coordinates
shops_df = pd.read_csv(os.path.join(silver_path, 'clean_outlet_coords.csv'))

# Load the raw scraped POIs
pois_df = pd.read_csv(os.path.join(bronze_path, 'external_scraped_pois.csv'))

# Clean up any POIs that might have missing coordinates just to be safe
pois_df = pois_df.dropna(subset=['Latitude', 'Longitude'])

print(f"Loaded {len(shops_df)} Shops and {len(pois_df)} POIs.")

# --- 2. THE SPATIAL MATH (BALL TREE ALGORITHM) ---
print("\nBuilding Spatial Tree for lightning-fast calculations...")

# Convert all Lat/Lon to Radians for the Haversine formula (Earth is a sphere)
shops_rad = np.deg2rad(shops_df[['Latitude', 'Longitude']])
pois_rad = np.deg2rad(pois_df[['Latitude', 'Longitude']])

# Build the BallTree using the POIs
tree = BallTree(pois_rad, metric='haversine')

# Earth's radius in kilometers
EARTH_RADIUS_KM = 6371.0

# Define our Catchment Radius: 1 kilometer
radius_km = 1.0
radius_rad = radius_km / EARTH_RADIUS_KM

print(f"Calculating POIs within a {radius_km}km radius for EVERY shop...")

# Query the tree: Find all POIs within the radius for every shop
# This returns a list of arrays (indices of the POIs near each shop)
indices_within_radius = tree.query_radius(shops_rad, r=radius_rad)

# --- 3. FEATURE ENGINEERING ---
print("Engineering specific POI features...")


schools_count = []
hospitals_count = []
transit_count = [] # Combining Bus Stops and Stations
markets_count = []

for i, nearby_poi_indices in enumerate(indices_within_radius):
    if len(nearby_poi_indices) == 0:
        # Shop is in the middle of nowhere
        schools_count.append(0)
        hospitals_count.append(0)
        transit_count.append(0)
        markets_count.append(0)
    else:
        # Get the types of POIs near this specific shop
        nearby_types = pois_df.iloc[nearby_poi_indices]['POI_Type'].values

        # Count them up
        schools_count.append(np.sum(nearby_types == 'School'))
        hospitals_count.append(np.sum(nearby_types == 'Hospital'))
        transit_count.append(np.sum((nearby_types == 'Bus_Stop') | (nearby_types == 'Bus_Station')))
        markets_count.append(np.sum(nearby_types == 'Marketplace'))

# Add these new engineered features back to our shops dataframe
shops_df['Catchment_Schools_1km'] = schools_count
shops_df['Catchment_Hospitals_1km'] = hospitals_count
shops_df['Catchment_Transit_1km'] = transit_count
shops_df['Catchment_Markets_1km'] = markets_count
shops_df['Total_POIs_1km'] = shops_df['Catchment_Schools_1km'] + shops_df['Catchment_Hospitals_1km'] + shops_df['Catchment_Transit_1km'] + shops_df['Catchment_Markets_1km']

# --- 4. SAVE TO GOLD LAYER ---
gold_export_path = os.path.join(gold_path, 'gold_outlet_catchment_features.csv')
shops_df.to_csv(gold_export_path, index=False)

print(f"\nSUCCESS! Gold features generated and saved to: {gold_export_path}")
display(shops_df.head(10))

Mounting Drive and Loading Data...
Mounted at /content/drive
Loaded 19760 Shops and 6684 POIs.

Building Spatial Tree for lightning-fast calculations...
Calculating POIs within a 1.0km radius for EVERY shop...
Engineering specific POI features...

SUCCESS! Gold features generated and saved to: /content/drive/MyDrive/Datastorm/Gold/gold_outlet_catchment_features.csv


,Outlet_ID,Latitude,Longitude,Catchment_Schools_1km,Catchment_Hospitals_1km,Catchment_Transit_1km,Catchment_Markets_1km,Total_POIs_1km
0,OUT_00001,7.089846,79.979055,0,0,0,0,0
1,OUT_00002,7.000558,80.012422,1,0,0,0,1
2,OUT_00003,6.806170,79.854547,0,0,0,0,0
3,OUT_00004,6.703533,79.806919,0,0,0,0,0
4,OUT_00005,7.186878,79.869831,1,0,0,0,1
5,OUT_00006,6.957117,79.977724,0,0,0,0,0
6,OUT_00007,6.925250,79.803979,0,0,0,0,0
7,OUT_00008,6.852307,79.829302,0,0,0,0,0
8,OUT_00009,7.116597,79.852009,0,0,0,0,0
9,OUT_00010,6.912578,79.862382,11,5,25,1,42
